Finding a function $u=\sin(\pi x)\sin(\pi y) + \sin(\frac{\pi}{\omega}x)\sin(\frac{\pi}{\omega}y)$ that looks the way I want in the unit square.

In [25]:
# Import any packages needed
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
import numpy as np

In [26]:
# Domain is unit square, create initial mesh
N = 64
mesh = Mesh(unit_square.GenerateMesh(maxh=1/N))
mesh.nv, mesh.ne
# Draw(mesh)

(4891, 9524)

In [27]:
# Working with H1-conforming piecewise linear finite elements
# boundary conditions are Dirichlet on the left and right and default=Neumann on the other 2 sides
fes = H1(mesh, order=1, dirichlet="left|right")
fes.ndof

# # Set test and trial spaces
u = fes.TrialFunction()
phi = fes.TestFunction()

# # Solution will be stored as a grid function
gfu = GridFunction(fes)
gfu.Set(1, BND) # this sets both Dirichlet boundary components to 1, might want to modify later
Draw(gfu)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [33]:
# Build a possible sine-combo
w = 10
s = CoefficientFunction(sin(pi*x)*sin(pi*y)+(1/w)*sin(w*pi*x)*sin(w*pi*y))
Draw(s,mesh, deformation=True)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [29]:
# Coefficient function for K(x,y)
Kxy = CoefficientFunction(1+x*x+y*y)
# Kxy = CoefficientFunction(3)
sigma = 0.5

# Bilinear form
a = BilinearForm(fes)
a += Kxy*grad(u)*grad(phi)*dx+sigma*sigma*u*phi*dx
a.Assemble()

# Right hand side
f = LinearForm(fes)
f.Assemble()

print(a.mat)
# print(f.vec)

Row 0:   0: 1.00006   4: -0.500025   255: -0.500025
Row 1:   1: 1.68999   66: -0.598377   67: -0.599953   318: -0.491641
Row 2:   2: 2.97923   129: -1.48961   130: -1.48961
Row 3:   3: 1.68968   192: -0.600783   193: -0.596169   443: -0.492714
Row 4:   0: -0.500025   4: 1.89651   5: -0.522677   255: -0.16056   256: -0.713212
Row 5:   4: -0.522677   5: 1.87782   6: -0.336178   256: -0.203359   257: -0.815572
Row 6:   5: -0.336178   6: 1.7473   7: -0.280076   257: -0.472855   258: -0.658164
Row 7:   6: -0.280076   7: 1.73979   8: -0.269134   258: -0.589405   259: -0.601151
Row 8:   7: -0.269134   8: 1.74415   9: -0.273173   259: -0.621481   260: -0.580336
Row 9:   8: -0.273173   9: 1.74845   10: -0.278263   260: -0.618871   261: -0.578121
Row 10:   9: -0.278263   10: 1.75355   11: -0.282712   261: -0.612852   262: -0.579694
Row 11:   10: -0.282712   11: 1.75963   12: -0.286646   262: -0.608352   263: -0.581896
Row 12:   11: -0.286646   12: 1.76667   13: -0.289983   263: -0.605374   264: 

In [34]:
q = GridFunction(fes)
q.Set(s) 
Draw(q, deformation=True)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [35]:
r = f.vec.CreateVector()
r.data = f.vec - a.mat * q.vec
currerr = r.Norm()
print(currerr)


2.858431162987392
